In [3]:
import torch.nn as nn
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
import torchvision.transforms as T
import torch


In [9]:
backbone = resnet_fpn_backbone('resnet50', weights=None)  # or weights='DEFAULT' if available
model_old = FasterRCNN(backbone, num_classes=4)  # 3 classes (brick kiln, background, and 2 more)
model_old

/opt/anaconda3/envs/rishabh_sat/lib/python3.12/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
          (relu

In [14]:
import torch.nn as nn
from collections import OrderedDict
from torchvision.models.detection import FasterRCNN
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform

class AEFBackbone(nn.Module):
    def __init__(self, in_ch=64, out_ch=256):
        super().__init__()
        self.out_channels = out_ch
        self.neck = nn.Sequential(
            nn.Conv2d(in_ch, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        f = self.neck(x)                     # [B, out_ch, H, W]
        return OrderedDict([("0", f)])       # single feature level

backbone = AEFBackbone(in_ch=64, out_ch=256)

anchor_gen = AnchorGenerator(
    sizes=((16, 32, 64, 128, 256),),         # tune for kiln scale
    aspect_ratios=((0.5, 1.0, 2.0),),
)

roi_pool = MultiScaleRoIAlign(
    featmap_names=["0"],
    output_size=7,
    sampling_ratio=2,
)

model = FasterRCNN(
    backbone,
    num_classes=4,
    rpn_anchor_generator=anchor_gen,
    box_roi_pool=roi_pool,
)

# IMPORTANT: 64-channel normalization
model.transform = GeneralizedRCNNTransform(
    min_size=800, max_size=1333,
    image_mean=[0.0]*64,   # or per-band means if you have them
    image_std=[1.0]*64,    # or per-band stds
)


model

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): AEFBackbone(
    (neck): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track

In [13]:
embeddind_path="/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/uttar_pradesh/train/embeddings/28.0144_79.4250.tif"
import rasterio
with rasterio.open(embeddind_path) as src:
    img = src.read()  # (C, H, W)
    img = torch.from_numpy(img).float()
    print(img.shape)



torch.Size([64, 131, 117])


In [15]:
import rasterio
import numpy as np

emb_path = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/uttar_pradesh/train/embeddings/28.0144_79.4250.tif"

with rasterio.open(emb_path) as src:
    # masked=True returns a NumPy MaskedArray, so nodata pixels are ignored
    arr = src.read(masked=True).astype("float64")   # shape: (C, H, W)

# Per-channel mean/std ignoring masked (nodata) values
means = np.ma.mean(arr, axis=(1, 2)).filled(np.nan)              # (C,)
stds  = np.ma.std(arr,  axis=(1, 2), ddof=0).filled(np.nan)      # (C,)


print("C,H,W:", arr.shape)
print("means (list) =", means.tolist())
print("stds  (list) =", stds.tolist())

C,H,W: (64, 131, 117)
means (list) = [-0.0035911769019974006, -0.17339349413352478, 0.1485572510463926, -0.0005493668021946379, 0.07717213553362164, 0.09276310929898007, 0.05444426448231594, 0.08168920126848796, -0.17561137079419653, 0.08689443633937409, 0.11803469185464263, 0.14352532625790732, -0.31055683295495934, 0.07704186160191064, 0.10493102336903222, 0.08612828820996352, -0.18656269874132805, -0.044660368669572605, -0.1623744502463822, -0.29264560637091075, -0.11272932414292128, 0.16995060396713524, 0.035691528815662316, 0.034018942016624316, 0.05587900030182974, 0.1657316407181846, 0.031618126791096365, 0.10160454056819597, 0.0256422449228878, 0.030068970730574202, 0.10267135108814911, -0.005207398413763011, 0.10547192202384491, -0.08018703114234693, 0.14034206101358096, 0.18732595639448704, -0.1673225689840529, -0.026469710724948706, 0.02280792846212262, 0.02803002436323024, 0.07784199959247512, -0.033300860060087416, -0.14868570664527087, 0.06994306456933834, 0.0403769206825

In [16]:
from pathlib import Path
import rasterio
import numpy as np
from tqdm import tqdm

root = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/uttar_pradesh/train/embeddings"

tifs = sorted(Path(root).glob("*.tif"))
if not tifs:
    raise FileNotFoundError(f"No .tif files in {root}")

# Discover channel count from the first file
with rasterio.open(tifs[0]) as src0:
    C = src0.count

sum_c     = np.zeros(C, dtype=np.float64)
sqsum_c   = np.zeros(C, dtype=np.float64)
count_c   = np.zeros(C, dtype=np.int64)

for p in tqdm(tifs, desc="Accumulating per-channel stats"):
    with rasterio.open(p) as src:
        arr = src.read(masked=True).astype("float64")  # (C,H,W) masked array
    for c in range(C):
        data = arr[c].compressed()   # flatten & drop masked (nodata)
        if data.size == 0:
            continue
        sum_c[c]   += data.sum()
        sqsum_c[c] += (data * data).sum()
        count_c[c] += data.size

# Avoid divide-by-zero if any channel had all nodata
valid = count_c > 0
means = np.zeros(C, dtype=np.float64)
stds  = np.ones(C,  dtype=np.float64)

means[valid] = sum_c[valid] / count_c[valid]
var          = sqsum_c[valid] / count_c[valid] - means[valid]**2
var          = np.clip(var, 0.0, None)  # numeric safety
stds[valid]  = np.sqrt(var)

print("means =", means.tolist())
print("stds  =", stds.tolist())

Accumulating per-channel stats: 100%|██████████| 9122/9122 [08:27<00:00, 17.97it/s]

means = [0.005512964153215181, -0.11987435268608083, 0.13552110180969382, 0.04784786663301407, 0.039084079586355513, 0.08357980166617966, 0.008079974805496184, 0.0804157833852248, -0.17536296401197532, 0.08583340626521094, 0.10395276978803457, 0.11037433299508735, -0.33883215166422137, 0.08552636509091865, 0.06621250816198408, 0.12473327781351583, -0.14065470558468945, 0.0006543347093933454, -0.159685918318685, -0.1970384268791541, -0.1681533071149419, 0.24337346841159954, 0.0888412732683545, 0.014617966503103246, 0.04530234609007981, 0.16549913808446257, 0.11093384229175621, 0.055849104083761925, 0.031484384604785524, 0.07066140694658107, 0.09669197879897373, 0.02510314766959483, 0.03795032831532969, -0.06087265903646453, 0.11940944196000249, 0.1488690945846687, -0.1864594590969006, 0.015199833351348235, 0.05158426028462877, -0.0015092413639521745, 0.08896527416569643, 0.024274000147048714, -0.1419422691469271, 0.1296167008787966, 0.028765311047575653, 0.09476422649372462, 0.069369747

In [19]:
#!/usr/bin/env python3
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm

# Set this to (800, 800) if you want to compute stats AFTER resizing
TARGET_SIZE = (800, 800)

# Your directory:
root = Path("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/iclr_2026_processed_data/final_data/uttar_pradesh/train/images")

# Try to import rasterio for GeoTIFFs (optional)
try:
    import rasterio
    HAS_RASTERIO = True
except Exception:
    HAS_RASTERIO = False

IMG_EXTS  = {".png", ".jpg", ".jpeg"}
TIF_EXTS  = {".tif", ".tiff"}

files = sorted([p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS | TIF_EXTS])
if not files:
    raise FileNotFoundError(f"No images found under: {root}")

# First pass: decide channel count
def infer_channels(path):
    ext = path.suffix.lower()
    if ext in IMG_EXTS:
        return 3  # we'll convert to RGB
    elif ext in TIF_EXTS and HAS_RASTERIO:
        with rasterio.open(path) as src:
            return src.count
    else:
        raise RuntimeError(f"Unsupported file or rasterio missing: {path}")

C = infer_channels(files[0])
sum_c   = np.zeros(C, dtype=np.float64)
sqsum_c = np.zeros(C, dtype=np.float64)
cnt_c   = np.zeros(C, dtype=np.int64)

def to_float01(arr):
    # arr expected in uint8 [0..255] or float [0..1]
    arr = arr.astype(np.float32, copy=False)
    if arr.max() > 1.0:
        arr /= 255.0
    return arr

for p in tqdm(files, desc="Computing stats"):
    ext = p.suffix.lower()
    if ext in IMG_EXTS:
        # Load as RGB (H,W,3)
        im = Image.open(p).convert("RGB")
        if TARGET_SIZE is not None:
            im = im.resize(TARGET_SIZE, Image.BILINEAR)
        arr = np.asarray(im)  # (H,W,3), uint8
        arr = to_float01(arr)
        # Per-channel reduce
        H, W, _ = arr.shape
        for c in range(3):
            x = arr[..., c].reshape(-1)
            sum_c[c]   += x.sum()
            sqsum_c[c] += (x * x).sum()
            cnt_c[c]   += x.size

    elif ext in TIF_EXTS:
        if not HAS_RASTERIO:
            raise RuntimeError(f"rasterio not available for {p}")
        with rasterio.open(p) as src:
            marr = src.read(masked=True)   # (C,H,W) masked array
        # Optional resize (per-band) using PIL for simplicity
        if TARGET_SIZE is not None:
            C_now, H, W = marr.shape
            resized = np.zeros((C_now, TARGET_SIZE[1], TARGET_SIZE[0]), dtype=np.float32)  # (C,H,W)
            mask_rs = np.zeros_like(resized, dtype=bool)
            for c in range(C_now):
                # Data
                band = marr[c]
                # Fill masked with 0, keep a separate mask
                band_data = band.filled(0).astype(np.float32)
                band_im = Image.fromarray(band_data)
                band_im = band_im.resize(TARGET_SIZE, Image.BILINEAR)
                resized[c] = np.asarray(band_im, dtype=np.float32)

                # Resize mask with nearest (True stays True)
                m_im = Image.fromarray((~band.mask).astype(np.uint8) * 255)
                m_im = m_im.resize(TARGET_SIZE, Image.NEAREST)
                mask_rs[c] = np.asarray(m_im, dtype=np.uint8) > 0
            marr = np.ma.array(resized, mask=~mask_rs)
        # Accumulate (mask-aware)
        for c in range(marr.shape[0]):
            data = marr[c].compressed().astype(np.float64)  # drop masked
            if data.size == 0:
                continue
            # If looks like uint8 range, map to [0,1]; otherwise assume already float
            if data.max() > 1.0:
                data = data / 255.0
            sum_c[c]   += data.sum()
            sqsum_c[c] += (data * data).sum()
            cnt_c[c]   += data.size

    else:
        # Should not happen due to filter above
        continue

# Finalize
valid = cnt_c > 0
means = np.zeros(C, dtype=np.float64)
stds  = np.ones(C,  dtype=np.float64)

means[valid] = sum_c[valid] / cnt_c[valid]
var = (sqsum_c[valid] / cnt_c[valid]) - (means[valid] ** 2)
var = np.clip(var, 0.0, None)
stds[valid] = np.sqrt(var)

print(f"Files processed: {len(files)}")
print(f"Channels: {C}")
print("means =", means.tolist())
print("stds  =", stds.tolist())

Computing stats: 100%|██████████| 9122/9122 [01:19<00:00, 114.07it/s]

Files processed: 9122
Channels: 3
means = [0.3099242753302991, 0.32896104479327426, 0.28209325850908495]
stds  = [0.15498155118861168, 0.14239670148542716, 0.14055865945267154]
